# GTEx computational timing

Wall time and peak resident memory for nine factorization methods on the GTEx v8 bulk matrix (21,613 genes x 17,382 samples, K = 578), three seeds each: **27 fits**. Unlike the pooled pseudobulk benchmark there is a single dataset here, so **each plotted observation is one fit**, not a sum across datasets.

**Excluded on purpose, not dropped.**

- **CoGAPS.** `rule cogaps_gtex` was run on GTEx and killed at its 7-day wall-clock budget without converging; the evidence file is printed below. That is the same reason CoGAPS is excluded from `rule full_models_gtex`. Its GTEx runtime is right-censored at >7 days (>10,080 min) and is reported as such rather than as a data point.
- **MOFA-FLEX base.** No prior-free MOFA-FLEX GTEx production model exists (`rule mofa_flex_base_gtex` fails on this dataset), so there is no published fit whose runtime could be measured.

**What is and is not in the clock.** Every fit reuses the preprocessed artifacts from `rule clamp_gtex`, so the ~48 min GCT parse / FBM build / preprocess / z-score / rsvd / K-estimation stage is excluded, matching the pseudobulk benchmark which reuses `rules.preprocess_pseudobulk.output`. Loading the input matrix *is* inside the clock.

**CLAMPfull includes CLAMPbase.** `CLAMPfull()` takes the fitted base model as `clamp.base.result`, so a CLAMPfull run necessarily refits CLAMPbase first. The two boxes are not independent: CLAMPfull is CLAMPbase plus the prior-informed refit.

**Seeds.** Seeds 123/456/789 vary stochastic initialisation where a method has any (the randomized PCA/ICA/NMF solvers, PLIER cross-validation, MOFA-FLEX). GSSig (`prcomp` plus Ward.D clustering) is deterministic given the data, so its three replicates measure run-to-run machine noise rather than seed sensitivity.


💡 **Environment:** `clamp-analyses`


In [ ]:
suppressPackageStartupMessages({
  library(data.table)
  library(ggplot2)
  library(yaml)
  library(here)
  library(scales)
})


In [ ]:
if (exists("snakemake")) {
  timing_paths <- as.character(snakemake@input[["timings"]])
  shape_path <- snakemake@input[["shape"]]
  gmt_path <- snakemake@input[["gmt"]]
  cogaps_evidence <- snakemake@input[["cogaps_evidence"]]
  expected_dataset <- as.character(snakemake@params[["dataset"]])
  expected_methods <- as.character(snakemake@params[["methods"]])
  expected_seeds <- as.integer(snakemake@params[["seeds"]])
  expected_threads <- as.integer(snakemake@params[["threads"]])
  out_long <- snakemake@output[["long"]]
  out_per_fit <- snakemake@output[["per_fit"]]
  out_summary <- snakemake@output[["summary"]]
  runtime_cfg <- yaml::read_yaml(here("workflow/config/runtime_benchmark_gtex.yaml"))$runtime_benchmark_gtex
} else {
  runtime_cfg <- yaml::read_yaml(here("workflow/config/runtime_benchmark_gtex.yaml"))$runtime_benchmark_gtex
  expected_dataset <- as.character(runtime_cfg$dataset)
  expected_methods <- unlist(runtime_cfg$methods, use.names = FALSE)
  expected_seeds <- as.integer(unlist(runtime_cfg$seeds, use.names = FALSE))
  expected_threads <- as.integer(runtime_cfg$threads)
  out_dir <- Sys.getenv("CLAMP_TIMING_ROOT", unset = here(runtime_cfg$output_root))
  timing_paths <- list.files(file.path(out_dir, "timings"), pattern = "[.]csv$",
                             recursive = TRUE, full.names = TRUE)
  shape_path <- file.path(out_dir, "matrix_shape.csv")
  gmt_path <- here(yaml::read_yaml(here("workflow/config/pseudobulk.yaml"))$references$go_bp_file)
  cogaps_evidence <- here(runtime_cfg$cogaps_evidence)
  out_long <- file.path(out_dir, "runtime_long.csv")
  out_per_fit <- file.path(out_dir, "runtime_per_fit.csv")
  out_summary <- file.path(out_dir, "runtime_summary.csv")
}

# The exclusions are a claim the report makes, so assert they hold and print why.
stopifnot(!("CoGAPS" %in% expected_methods))
stopifnot(!("MOFA-FLEX_base" %in% expected_methods))
stopifnot(length(expected_methods) == 9L, length(expected_seeds) == 3L)

for (nm in names(runtime_cfg$excluded_methods)) {
  cat("EXCLUDED  ", nm, ": ", trimws(runtime_cfg$excluded_methods[[nm]]), "\n\n", sep = "")
}
cat("CoGAPS evidence (", cogaps_evidence, "):\n  ",
    paste(readLines(cogaps_evidence), collapse = "\n  "), "\n\n", sep = "")

expected_runs <- length(expected_methods) * length(expected_seeds)
stopifnot(expected_runs == 27L)
stopifnot(length(timing_paths) == expected_runs)


## Validate and aggregate all 27 fits


In [ ]:
runtime_long <- rbindlist(lapply(timing_paths, fread), use.names = TRUE, fill = TRUE)
required_columns <- c("dataset", "method", "seed", "threads", "n_genes", "n_samples",
                      "elapsed_seconds", "started_at", "finished_at", "command",
                      "status", "exit_code",
                      "peak_rss_mb", "peak_rss_source", "user_cpu_seconds",
                      "system_cpu_seconds", "cpu_utilization",
                      "started_epoch", "finished_epoch", "term_signal", "output_bytes")
stopifnot(all(required_columns %in% names(runtime_long)))
stopifnot(nrow(runtime_long) == expected_runs)

# One dataset, so a fit is identified by (method, seed) alone: there is no
# cross-dataset coverage check as in the pooled pseudobulk report.
stopifnot(uniqueN(runtime_long, by = c("method", "seed")) == expected_runs)
stopifnot(setequal(runtime_long$dataset, expected_dataset))
stopifnot(setequal(runtime_long$method, expected_methods))
stopifnot(setequal(as.integer(runtime_long$seed), expected_seeds))
stopifnot(runtime_long[, .N, by = method][, all(N == length(expected_seeds))])
stopifnot(all(runtime_long$threads == expected_threads))
stopifnot(all(runtime_long$status == "success"), all(runtime_long$exit_code == 0L))
stopifnot(all(runtime_long$term_signal == 0L))
stopifnot(all(runtime_long$output_bytes > 0))
stopifnot(all(is.finite(runtime_long$elapsed_seconds)), all(runtime_long$elapsed_seconds > 0))
stopifnot(all(is.finite(runtime_long$peak_rss_mb)), all(runtime_long$peak_rss_mb > 0))
stopifnot(all(runtime_long$peak_rss_mb < 125 * 1024))
stopifnot(all(runtime_long$peak_rss_source == "os.wait4:ru_maxrss"))

# Every fit saw the same matrix.
gtex_shape <- fread(shape_path)
stopifnot(uniqueN(runtime_long$n_genes) == 1L, uniqueN(runtime_long$n_samples) == 1L)
stopifnot(runtime_long$n_genes[1] == gtex_shape$n_genes[1],
          runtime_long$n_samples[1] == gtex_shape$n_samples[1])
cat(sprintf("GTEx matrix: %d genes x %d samples\n",
            runtime_long$n_genes[1], runtime_long$n_samples[1]))

# Every MOFA-FLEX fit saw the same pathway prior, and it is the same pinned GO:BP
# file CLAMP and PLIER are trained against.  rule pathway_prior already verifies this
# checksum on download, so re-checking it here proves the prior did not drift between
# that rule and these fits.  This replaced a post-hoc fingerprint of a downloaded
# MSigDB cache, which was weaker: it recorded whatever had been fetched rather than
# enforcing a pin.
gmt_sha256 <- yaml::read_yaml(here("workflow/config/pseudobulk.yaml"))$references$go_bp_sha256
observed_sha256 <- tools::sha256sum(gmt_path)[[1]]
stopifnot(identical(observed_sha256, gmt_sha256))
cat(sprintf("Pathway prior: %s\n  sha256 = %s (matches the pinned checksum)\n",
            basename(gmt_path), observed_sha256))

# Serialization audit.  The benchmark claims exactly one fit ran at a time.  Prove
# it from the records rather than trusting that `--resources timing_slot=1` was
# actually passed: Snakemake treats a resource not named there as unlimited, so
# omitting the flag would silently parallelise the sweep and inflate every timing.
ord <- order(runtime_long$started_epoch)
starts <- runtime_long$started_epoch[ord]
ends <- runtime_long$finished_epoch[ord]
stopifnot(all(ends[-length(ends)] <= starts[-1] + 1e-6))
cat(sprintf("Serial execution confirmed: %d non-overlapping intervals spanning %.2f days\n",
            length(starts), (max(ends) - min(starts)) / 86400))

runtime_per_fit <- runtime_long[, .(
  method, seed,
  elapsed_seconds,
  elapsed_minutes = elapsed_seconds / 60,
  elapsed_hours = elapsed_seconds / 3600,
  peak_rss_mb,
  peak_rss_gb = peak_rss_mb / 1024,
  cpu_utilization,
  n_genes, n_samples
)]
stopifnot(nrow(runtime_per_fit) == expected_runs)

runtime_summary <- runtime_per_fit[, .(
  n_seeds = .N,
  median_minutes = median(elapsed_minutes),
  q1_minutes = as.numeric(quantile(elapsed_minutes, 0.25)),
  q3_minutes = as.numeric(quantile(elapsed_minutes, 0.75)),
  min_minutes = min(elapsed_minutes),
  max_minutes = max(elapsed_minutes),
  median_peak_rss_gb = median(peak_rss_gb),
  min_peak_rss_gb = min(peak_rss_gb),
  max_peak_rss_gb = max(peak_rss_gb),
  median_cpu_utilization = median(cpu_utilization)
), by = method]
stopifnot(nrow(runtime_summary) == length(expected_methods))
stopifnot(all(runtime_summary$n_seeds == length(expected_seeds)))

# One shared method order (ascending median wall time) for BOTH plots, so the
# timing and memory panels are read against the same x axis.
timing_method_order <- runtime_summary[order(median_minutes), method]
runtime_per_fit[, method := factor(method, levels = timing_method_order)]
setorder(runtime_per_fit, method, seed)
setorder(runtime_summary, median_minutes)

for (path in c(out_long, out_per_fit, out_summary)) {
  dir.create(dirname(path), recursive = TRUE, showWarnings = FALSE)
}
fwrite(runtime_long, out_long)
fwrite(runtime_per_fit, out_per_fit)
fwrite(runtime_summary, out_summary)
runtime_summary[]


## Wall time per fit

Five caveats to read the figure against.

1. **CLAMPfull includes CLAMPbase**, since the full fit takes the base model as input.
2. **Not every method can use the four threads it was given.** All nine fits were pinned to `threads: 4` via the usual OMP/MKL/OPENBLAS environment variables, and the `cpu_utilization` column records what each actually achieved. The Python methods and CLAMP reach 3.3 to 3.7, so the cap genuinely bound. PLIER and Flashier sit at ~1.0: their dominant loops (PLIER's cross-validation, Flashier's backfit) are serial R, so the BLAS cap never binds and their wall time is effectively single-core. Read their bars as "this is what the method costs as implemented", not as "this is what it costs on four cores". The `median_cpu_utilization` column of `runtime_summary.csv` carries this per method.
3. **The methods do not all load the matrix the same way.** The R methods `readRDS` a 2.9 GB `.rds`, the Python methods (PCA, NMF, ICA, MOFA-FLEX) `pd.read_csv` a 6.8 GB `.csv`, and both are inside the clock. That is a systematic few-minute I/O tax on the Python methods which does not exist in the pseudobulk benchmark, where every method reads the same CSV. It is not corrected here, because converting formats would change what the published pipeline actually does.
4. **n = 3**, so the boxplot hinges are the extremes. The geometry is kept identical to the pseudobulk panel for visual consistency.
5. **These are fixed iteration budgets, not convergence points.** Flashier is capped at 20 backfit iterations and MOFA-FLEX at 1000 epochs, matched to the pseudobulk pipeline. Both sit below their package defaults, which are not computationally practical on this matrix, and neither method reaches its convergence criterion at any cap we have run: Flashier stops with its between-iteration criterion orders of magnitude above tolerance, and MOFA-FLEX consumes its full epoch budget without the built-in early stopper ever firing. Read the bars as cost under an equal budget, not as cost to converge.
6. **CoGAPS would be off-scale** at >10,080 min and is absent for the reason printed above.


In [ ]:
figure_cfg <- yaml::read_yaml(here("config.yaml"))
method_colors <- unlist(figure_cfg$MODEL_COLORS)
names(method_colors)[names(method_colors) == "GenomicSuperSignature"] <- "GSSig"

runtime_plot <- ggplot(runtime_per_fit, aes(method, elapsed_minutes, fill = method)) +
  geom_boxplot(width = 0.58, outlier.shape = NA, color = "black", linewidth = 0.35) +
  geom_point(aes(shape = factor(seed)), position = position_jitter(width = 0.08, height = 0),
             size = 2.2, fill = "white", color = "black", stroke = 0.5) +
  scale_fill_manual(values = method_colors, na.value = "grey70", guide = "none") +
  scale_shape_manual(values = c(`123` = 21, `456` = 22, `789` = 24), name = "Seed") +
  scale_y_log10(breaks = scales::log_breaks(n = 6), labels = scales::label_number()) +
  labs(x = NULL, y = "Wall time per GTEx fit (min)") +
  theme_classic(base_size = 9, base_family = "Helvetica") +
  theme(axis.text.x = element_text(angle = 40, hjust = 1),
        legend.position = "top", legend.title = element_text(face = "plain"))

options(repr.plot.width = 7.2, repr.plot.height = 3.15)
runtime_plot


## Peak resident memory per fit

`peak_rss_mb` is `ru_maxrss` from `os.wait4` on the single timed child process, that is the maximum resident set size reached by that process and any descendants it reaped. Two things to keep in mind: it is a maximum over the process tree rather than a sum, which is exact for these nine single-process methods; and for CLAMPbase, CLAMPfull and PLIER it includes resident pages of the memory-mapped 3.0 GB FBM backing file, which is genuine residency but page-cache dependent and so slightly less reproducible than for the purely in-memory methods.

The methods keep the same left-to-right order as the timing figure (ascending median wall time), so the two panels can be read together: the cost of a method is the pair, not either number alone.


In [ ]:
memory_plot <- ggplot(runtime_per_fit, aes(method, peak_rss_gb, fill = method)) +
  geom_boxplot(width = 0.58, outlier.shape = NA, color = "black", linewidth = 0.35) +
  geom_point(aes(shape = factor(seed)), position = position_jitter(width = 0.08, height = 0),
             size = 2.2, fill = "white", color = "black", stroke = 0.5) +
  geom_hline(yintercept = 125, linetype = "dashed", colour = "grey40", linewidth = 0.35) +
  annotate("text", x = 0.6, y = 125, label = "machine RAM (125 GB)",
           hjust = 0, vjust = -0.6, size = 2.4, colour = "grey40") +
  scale_fill_manual(values = method_colors, na.value = "grey70", guide = "none") +
  scale_shape_manual(values = c(`123` = 21, `456` = 22, `789` = 24), name = "Seed") +
  scale_y_continuous(limits = c(0, NA), expand = expansion(mult = c(0, 0.12))) +
  labs(x = NULL, y = "Peak resident memory per GTEx fit (GB)") +
  theme_classic(base_size = 9, base_family = "Helvetica") +
  theme(axis.text.x = element_text(angle = 40, hjust = 1),
        legend.position = "top", legend.title = element_text(face = "plain"))

options(repr.plot.width = 7.2, repr.plot.height = 3.15)
memory_plot
